# ⚠️ Aviso Importante - Conexões com Banco de Dados

![Jupyter](https://img.shields.io/badge/Jupyter-111827?style=flat-square&logo=jupyter&logoColor=F37626)
![Python](https://img.shields.io/badge/Python-111827?style=flat-square&logo=python&logoColor=3776AB)
![Python Version](https://img.shields.io/badge/python-3.14+-blue)
![Tópico](https://img.shields.io/badge/tópico-boas%20práticas%20%7C%20segurança-teal)
![Dificuldade](https://img.shields.io/badge/dificuldade-Iniciante-green)
![Pré-req](https://img.shields.io/badge/pré--req-pyodbc-purple)
![Biblioteca](https://img.shields.io/badge/requer-pyodbc-orange)

> Antes de escrever o primeiro `CREATE`/`UPDATE`/`DELETE` de verdade: banco de dados é um recurso compartilhado, muitas vezes em produção. Um erro aqui não é "abrir o arquivo errado" — é **derrubar conexão de outras pessoas**, **deixar dado corrompido** ou **vazar credencial**. Este notebook é só de leitura, sem exercício — são 4 cuidados que valem pra qualquer banco, não só SQL Server.

## 📋 Conteúdo

1. [Nunca Deixe Credencial no Código](#-1-nunca-deixe-credencial-no-código)
2. [Sempre Feche a Conexão](#-2-sempre-feche-a-conexão)
3. [Nunca Monte SQL Concatenando Texto](#-3-nunca-monte-sql-concatenando-texto)
4. [Commit Não é Automático](#-4-commit-não-é-automático)


<h2 align="left">🔑 1. <mark style="background-color: white; color: red">NUNCA</mark> Deixe Credencial no Código</h2>

Usuário e senha do banco não devem aparecer direto no notebook nem serem commitados no Git. O jeito mais simples de resolver isso — usado no restante deste módulo — é isolar essas informações num arquivo separado (`conexao.py`), importado como qualquer outro módulo.

| Errado ❌ | Certo ✅ |
|---|---|
| `pyodbc.connect("...PWD=minha_senha_real;...")` espalhado em várias células | `from conexao import nova_conexao_sqlserver` |

> 💡 Em um projeto profissional, esse arquivo nem ficaria no Git — as credenciais viriam de variáveis de ambiente ou de um cofre de segredos (Azure Key Vault, AWS Secrets Manager etc.). Aqui, por ser um banco local de estudo, mantemos tudo isolado em `conexao.py`.

<h2 align="left">🔒 2. <mark style="background-color: white; color: green">SEMPRE</mark> Feche a Conexão</h2>

Cada conexão aberta consome um "slot" no servidor. Esquecer de fechar (ou travar no meio de um erro sem fechar) deixa conexões penduradas, e o banco tem um limite de conexões simultâneas.

| Padrão 🔑 | Por que usar 🔓 |
|---|---|
| `conexao.close()` no final | Garante que o slot é liberado |
| `with pyodbc.connect(...) as conexao:` | Fecha sozinho, mesmo se o código der erro no meio |

```python
# withpadrão mais seguro que abrir/fechar manualmente
with pyodbc.connect(string_conexao) as conexao:
    cursor = conexao.cursor()
    cursor.execute("SELECT 1")
    print(cursor.fetchone())
# a conexão já fechou sozinha aqui fora
```

<h2 align="left">💉 3. <mark style="background-color: white; color: red">NUNCA</mark> Monte SQL Concatenando Texto</h2>

Colar um valor direto dentro da string SQL (com f-string ou `+`) abre brecha pra **SQL Injection** — alguém pode digitar um valor que muda o comando executado. A solução é sempre usar **parâmetros** (`?`), deixando o driver escapar o valor com segurança.

| Errado ❌ | Certo ✅ |
|---|---|
| `cursor.execute(f"SELECT * FROM Clientes WHERE Nome = '{nome}'")` | `cursor.execute("SELECT * FROM Clientes WHERE Nome = ?", nome)` |

> ⚠️ **Atenção:** se `nome` fosse literalmente `' OR '1'='1`, a versão concatenada viraria `WHERE Nome = '' OR '1'='1'` — e devolveria a tabela inteira. Com `?`, o valor nunca é interpretado como parte do comando SQL.

<h2 align="left"> 💾 4. Commit <mark style="background-color: white; color: red">NÃO É</mark> Automático</h2>

Por padrão, `pyodbc` abre a conexão em modo transacional: um `INSERT`/`UPDATE`/`DELETE` só é gravado de verdade depois de um `conexao.commit()`. Sem isso, a mudança fica pendente — e se a conexão fechar antes, ela é descartada.

| Comando 🔑 | O que faz 🔓 |
|---|---|
| `conexao.commit()` | Confirma (grava definitivamente) todas as mudanças pendentes |
| `conexao.rollback()` | Desfaz as mudanças pendentes, como se nunca tivessem acontecido |
| `pyodbc.connect(..., autocommit=True)` | Cada comando já é gravado sozinho, sem precisar de `commit()` manual |

> 💡 **Por que não usar `autocommit=True` sempre?** Porque às vezes você quer que várias mudanças aconteçam **juntas ou nenhuma delas** (ex.: descontar estoque de 3 insumos ao mesmo tempo) — se uma falhar no meio, o `rollback()` desfaz tudo, e o banco nunca fica num estado "pela metade".

Com esses 4 cuidados em mente, o próximo notebook já é o passo a passo prático completo — conectar, criar, ler, atualizar, apagar e fechar, tudo junto.

> ▶️ Próximo notebook: **Passo a Passo do pyodbc**.